In [1]:
import os
import subprocess
import sys
from contextlib import ExitStack
from time import perf_counter

from IPython.display import display
from qdk.qre import ProfilerResult
from qdk.qre.application import QspApplication

app = QspApplication.for_cpp_code("""
#include "qep_interop.h"

// Execute the same 100 million loop gates using an in-process profiler.
int main() {
    qdk_init("counting_profiler");
    qdk_h(0);
    const size_t start = qdk_mz_reset(0);
    if (start > 1) return 1;
    for (size_t i = start; i < start + 1000000; ++i) {
        for (size_t j = 0; j < 50; ++j) {
            qdk_h(j);
            qdk_t(j);
        }
    }
    qdk_reset(0);
    const int status = qdk_mz(0) > 1;
    qdk_save_profile("profile.json");
    return status;
}
""")
started = perf_counter()
executable = app.compile()
compiled = perf_counter()
with ExitStack() as cleanup:
    subprocess.run([str(executable)], cwd=executable.parent, check=True)
finished = perf_counter()
result = ProfilerResult.from_json((executable.parent / "profile.json").read_text(encoding="utf-8"))
elapsed = perf_counter() - started
count = result.totals.total_operations_count
assert count == 100_000_004
assert result.totals.calls_per_operation == {
    "h": 50_000_001, "t": 50_000_000, "mz_reset": 1, "reset": 1, "mz": 1
}
assert result.totals.logical_qubits == 50
display(result)
print(f"{count:,} ops in {elapsed:.3f}s (including compilation, startup, and profile I/O)")
print(f"{count / elapsed / 1e6:.3f} million ops/s, {1e9 * elapsed / count:.1f} ns/op")
print(f"Excluding compilation and result loading: {1e9 * (finished - compiled) / count:.1f} ns/op")

Profile saved to profile.json


ProfilerResult { totals: TotalCounts { logical_qubits: 50, logical_compute_qubits: 50, physical_qubits: 50, total_runtime_sec: 0.0, total_operations_count: 100000004, calls_per_operation: {"t": 50000000, "mz_reset": 1, "h": 50000001, "reset": 1, "mz": 1}, runtime_sec_per_operation: {"t": 0.0, "mz_reset": 0.0, "h": 0.0, "reset": 0.0, "mz": 0.0}, error: 0.0 }, time_series_count: 0, aggregation_window_sec: None, infos: [] }

100,000,004 ops in 3.100s (including compilation, startup, and profile I/O)
32.259 million ops/s, 31.0 ns/op
Excluding compilation and result loading: 29.1 ns/op
